In [1]:
import csv
from collections import Counter

rows = list(csv.DictReader(open('/mnt/results/hdna_benchmark_provenance_audit.csv')))
assert len(rows) == 80

# --- Curation decisions (approved plan, Step 1) ---
REMOVE = {  # no paper provenance
    'HDNA0035': "Removed (no provenance): the 42-nt oligo = pGG32 core + EcoRI/BamHI overhangs; only the 32-nt core is published (Belotserkovskii 1990, already present as HDNA0002). Near-duplicate of HDNA0002.",
    'HDNA0036': "Removed (no provenance): pMycAG-derived 22-nt mutant not present in Belotserkovskii 2007 (constructs were WT/TT/Q/AG/AA); no primary source found.",
    'HDNA0037': "Removed (no provenance): Mycm1+G21T derivative; no primary source found in any traced paper.",
}
EXCLUDE = {  # provenance exists, but invalid as an ACGT benchmark record
    'HDNA0026': "Excluded (invalid record): the non-forming label belongs to an inosine-substituted oligo (Del Mundo 2017); the FASTA silently converts inosine->G, manufacturing a perfect mirror repeat that was never experimentally tested. Produced 1 of 2 false positives.",
    'HDNA0080': "Excluded (label-scope error): 'forming' refers to an intermolecular TFO complex; the tool predicts intramolecular H-DNA only. Manufactured a false negative.",
}
LABEL_CONFLICT = {
    'HDNA0033': "Rooney & Moore 1995 report the AG32 H-DNA transition as strongly impaired, conflicting with the CSV 'forming' label. Kept per provenance rule; WS2 sensitivity analysis must report metrics with both labelings.",
}

out_rows = []
for r in rows:
    rid = r['record_id']
    if rid in REMOVE:
        r['curation_decision'] = 'removed'
        r['curation_justification'] = REMOVE[rid]
        r['label_conflict'] = 'false'
    elif rid in EXCLUDE:
        r['curation_decision'] = 'excluded'
        r['curation_justification'] = EXCLUDE[rid]
        r['label_conflict'] = 'false'
    else:
        r['curation_decision'] = 'kept'
        if rid in LABEL_CONFLICT:
            r['curation_justification'] = 'Kept (provenance verified). ' + LABEL_CONFLICT[rid]
            r['label_conflict'] = 'true'
        elif r['verification_status'] == 'citation_corrected':
            r['curation_justification'] = 'Kept: primary-source provenance verified after citation correction (corrected_pmid/corrected_doi columns).'
            r['label_conflict'] = 'false'
        else:
            r['curation_justification'] = 'Kept: primary-source provenance verified.'
            r['label_conflict'] = 'false'
    out_rows.append(r)

kept = [r for r in out_rows if r['curation_decision'] == 'kept']
print('Total rows written:', len(out_rows))
print('Kept:', len(kept), Counter(r['label'] for r in kept))
print('Removed:', [r['record_id'] for r in out_rows if r['curation_decision']=='removed'])
print('Excluded:', [r['record_id'] for r in out_rows if r['curation_decision']=='excluded'])
print('Label conflicts flagged:', [r['record_id'] for r in kept if r['label_conflict']=='true'])

fieldnames = list(out_rows[0].keys())
with open('/mnt/results/hdna_benchmark_experimental_v3.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    w.writerows(out_rows)
print('Wrote /mnt/results/hdna_benchmark_experimental_v3.csv')

Total rows written: 80
Kept: 75 Counter({'forming': 69, 'non_forming': 6})
Removed: ['HDNA0035', 'HDNA0036', 'HDNA0037']
Excluded: ['HDNA0026', 'HDNA0080']
Label conflicts flagged: ['HDNA0033']
Wrote /mnt/results/hdna_benchmark_experimental_v3.csv


In [2]:
cd /mnt/results && python3 generate_synthetic_negatives.py

eligible perturbation sources: 41


wrote /mnt/results/hdna_benchmark_balanced_v3.csv: 138 rows (75 experimental + 63 synthetic)


label balance: {'forming': 69, 'non_forming': 69}


per-class summary (strong mirror = arm>=8):


  mirror_disrupted_mutant      n=20  strong_mirror_id max=0.78  purine range=[0.16,0.96]


...

  random_gc_length_matched     n=12  strong_mirror_id max=0.78  purine range=[0.21,0.76]


  other_structure              n=12  strong_mirror_id max=1.00  purine range=[0.44,0.86]


  homopolymer_AT               n= 4  strong_mirror_id max=1.00  purine range=[0.00,1.00]


  perfect_mirror_balanced      n= 3  strong_mirror_id max=1.00  purine range=[0.46,0.50]


H-DNA-competent synthetic rows (must be []): []


In [3]:
import csv
from collections import Counter

path = '/mnt/results/hdna_benchmark_experimental_v3.csv'
rows = list(csv.DictReader(open(path)))
fieldnames = list(rows[0].keys())

LOW_JUST = {
    'HDNA0032': "Removed (low confidence tier): author decision to exclude all confidence_tier=low "
                "records, despite resolved provenance (exact 30-nt GG32 match, Rooney & Moore 1995, PMID 7892237).",
    'HDNA0033': "Removed (low confidence tier): author decision to exclude all confidence_tier=low records. "
                "Also carried the unresolved label conflict (Rooney & Moore 1995 report AG32 H-DNA transition "
                "strongly impaired vs CSV 'forming'); removal moots decision DEC-1.",
    'HDNA0034': "Removed (low confidence tier): author decision to exclude all confidence_tier=low records, "
                "despite resolved provenance (Belotserkovskii 2007 JBC 'AA' construct, PMID 17785457); "
                "removal moots the DEC-4 figure-level confirmation.",
}
n_changed = 0
for r in rows:
    if r['record_id'] in LOW_JUST:
        assert r['confidence_tier'] == 'low' and r['curation_decision'] == 'kept'
        r['curation_decision'] = 'removed'
        r['curation_justification'] = LOW_JUST[r['record_id']]
        n_changed += 1
assert n_changed == 3

staging = '/workspace/hdna_benchmark_experimental_v3.csv'
with open(staging, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    w.writerows(rows)

kept = [r for r in rows if r['curation_decision'] == 'kept']
print('staged', staging)
print('kept:', len(kept), Counter(r['label'] for r in kept))
print('kept confidence tiers:', Counter(r['confidence_tier'] for r in kept))
print('removed now:', sorted(r['record_id'] for r in rows if r['curation_decision']=='removed'))
print('excluded:', sorted(r['record_id'] for r in rows if r['curation_decision']=='excluded'))

staged /workspace/hdna_benchmark_experimental_v3.csv
kept: 72 Counter({'forming': 66, 'non_forming': 6})
kept confidence tiers: Counter({'high': 42, 'literature_curated': 25, 'moderate': 5})
removed now: ['HDNA0032', 'HDNA0033', 'HDNA0034', 'HDNA0035', 'HDNA0036', 'HDNA0037']
excluded: ['HDNA0026', 'HDNA0080']


In [4]:
cd /mnt/results && python3 generate_synthetic_negatives.py

experimental: 72 kept = 66 forming / 6 non-forming -> generating 60 synthetic negatives


eligible perturbation sources: 38


(staged via /workspace/hdna_benchmark_balanced_v3.csv — mount blocked direct overwrite)


wrote /mnt/results/hdna_benchmark_balanced_v3.csv: 132 rows (72 experimental + 60 synthetic)


label balance: {'forming': 66, 'non_forming': 66}


...

  random_gc_length_matched     n=11  strong_mirror_id max=0.78  purine range=[0.18,0.84]


  other_structure              n=12  strong_mirror_id max=1.00  purine range=[0.46,0.86]


  homopolymer_AT               n= 4  strong_mirror_id max=1.00  purine range=[0.00,1.00]


  perfect_mirror_balanced      n= 3  strong_mirror_id max=1.00  purine range=[0.46,0.58]


H-DNA-competent synthetic rows (must be []): []


In [5]:
import csv, json
from collections import Counter

def rc(s): return s.translate(str.maketrans('ACGT','TGCA'))[::-1]

exp = list(csv.DictReader(open('/mnt/results/hdna_benchmark_experimental_v3.csv')))
bal = list(csv.DictReader(open('/mnt/results/hdna_benchmark_balanced_v3.csv')))
kept = [r for r in exp if r['curation_decision']=='kept']

print('========== EXPERIMENTAL v3 (updated) ==========')
print('total rows in file:', len(exp))
print('kept:', len(kept), '| removed:', sum(r['curation_decision']=='removed' for r in exp),
      '| excluded:', sum(r['curation_decision']=='excluded' for r in exp))
print('kept labels:', dict(Counter(r['label'] for r in kept)))
print('kept tiers:', dict(Counter(r['confidence_tier'] for r in kept)))
assert not any(r['confidence_tier']=='low' for r in kept)
print('kept dispositions:', dict(Counter(r['disposition'] for r in kept)))
print('kept verification:', dict(Counter(r['verification_status'] for r in kept)))
print('kept studies:', len(set(r['study_id'] for r in kept)),
      '| kept families:', len(set(r['family_id'] for r in kept)))
print('label conflicts among kept:', [r['record_id'] for r in kept if r['label_conflict']=='true'])

print()
print('========== BALANCED v3 (regenerated) ==========')
syn = [r for r in bal if r['record_type']=='synthetic']
exp_in_bal = [r for r in bal if r['record_type']=='experimental']
print('total rows:', len(bal), '=', len(exp_in_bal), 'experimental +', len(syn), 'synthetic')
lab = Counter(r['label'] for r in bal)
print('labels:', dict(lab), f"-> {lab['forming']/len(bal):.1%} forming / {lab['non_forming']/len(bal):.1%} non-forming")
assert lab == {'forming':66,'non_forming':66}
assert len(exp_in_bal) == 72 and len(syn) == 60

# integrity checks
assert all(r['generation_method'] and r['seed']=='20260922' and r['expected_label_rationale'] for r in syn)
removed_ids = {r['record_id'] for r in exp if r['curation_decision']!='kept'}
assert not any(r['source_record'] in removed_ids for r in syn), 'synthetic derived from removed record!'
seqs = [r['sequence_5to3'] for r in bal]
assert len(set(seqs)) == len(seqs)
canon = set()
for s in seqs:
    c = min(s, rc(s)); assert c not in canon; canon.add(c)
bad = [r['record_id'] for r in syn
       if r['generation_method'] not in ('perfect_mirror_balanced','homopolymer_AT')
       and float(r['validation_strong_mirror_identity']) >= 0.80
       and int(r['validation_strong_mirror_arm']) >= 8
       and float(r['validation_purine_fraction']) >= 0.85]
assert bad == []
print('integrity: all synthetic have method+seed+rationale; none derived from removed/excluded records;')
print('no exact or reverse-complement duplicates; zero H-DNA-competent synthetic rows')

print()
print('synthetic class mix:')
for m, n in sorted(Counter(r['generation_method'].split(':')[0] for r in syn).items()):
    rs = [r for r in syn if r['generation_method'].split(':')[0]==m]
    sm = max(float(r['validation_strong_mirror_identity']) for r in rs)
    print(f'  {m:28s} n={n:2d}  max strong-mirror identity={sm:.2f}')
print()
print('experimental-only ratio: 66:6 = 11.0:1 (dual reporting still required)')
print('balanced ratio: 66:66 = 1:1')
print('ALL CHECKS PASSED')

========== EXPERIMENTAL v3 (updated) ==========
total rows in file: 80
kept: 72 | removed: 6 | excluded: 2
kept labels: {'forming': 66, 'non_forming': 6}
kept tiers: {'high': 42, 'moderate': 5, 'literature_curated': 25}
kept dispositions: {'primary': 60, 'secondary': 12}
kept verification: {'verified_consistent': 50, 'verified_exact': 1, 'citation_corrected': 21}
kept studies: 27 | kept families: 50
label conflicts among kept: []

========== BALANCED v3 (regenerated) ==========
total rows: 132 = 72 experimental + 60 synthetic
labels: {'forming': 66, 'non_forming': 66} -> 50.0% forming / 50.0% non-forming
integrity: all synthetic have method+seed+rationale; none derived from removed/excluded records;
no exact or reverse-complement duplicates; zero H-DNA-competent synthetic rows

synthetic class mix:
  dinucleotide_shuffle         n=10  max strong-mirror identity=0.78
  homopolymer_AT               n= 4  max strong-mirror identity=1.00
  mirror_disrupted_mutant      n=20  max strong-mirr

In [6]:
import re

path = '/mnt/results/HSeeker_Reviewer_Answer_Plan_v2.md'
text = open(path).read()
orig = text

# 1. Header: audit CSV reference -> experimental v3
text = text.replace(
    'a provenance audit of the benchmark executed alongside this plan (`hdna_benchmark_provenance_audit.csv`)',
    'a provenance audit and curation of the benchmark executed alongside this plan (`hdna_benchmark_experimental_v3.csv`)'
)

# 2. Part A intro: same reference
text = text.replace(
    'Full per-record results: `hdna_benchmark_provenance_audit.csv` (80 rows;',
    'Full per-record results: `hdna_benchmark_experimental_v3.csv` (80 rows; 72 kept, 6 removed, 2 excluded;'
)

# 3. A.1 orphan table: update HDNA0032-34 dispositions to removed
text = text.replace(
    '| HDNA0032 | GG32 (30-nt) | **Rooney & Moore 1995 PNAS** GG32. Their XY32 motif is 30 nt (3-G runs); template X=G,Y=G reproduces the sequence exactly. Distinct from the 32-nt pGG32 (Belotserkovskii 1990 / Wang 2004, 4-G runs). | verified_exact | **primary** (correct citation: PMID 7892237, DOI 10.1073/pnas.92.6.2141) |',
    '| HDNA0032 | GG32 (30-nt) | **Rooney & Moore 1995 PNAS** GG32 (exact match). | verified_exact | **removed** (low confidence tier; author decision to exclude all low-tier records despite resolved provenance). |'
)
text = text.replace(
    '| HDNA0033 | AG32 (30-nt) | **Rooney & Moore 1995** AG32 (exact template match, X=A,Y=G). | **label_conflict** | **secondary** — the only direct evidence for this exact construct is *negative* (no recombination stimulation; "transition much more difficult"). The CSV\'s `forming` label came from the unavailable document. Authors must relabel (non-forming, citing Rooney 1995) or exclude; report metrics both ways. |',
    '| HDNA0033 | AG32 (30-nt) | **Rooney & Moore 1995** AG32 (exact match). | **label_conflict** | **removed** (low confidence tier; author decision. Also carried the unresolved label conflict — Rooney 1995 report AG32 H-DNA transition strongly impaired vs CSV \'forming\'; removal moots DEC-1). |'
)
text = text.replace(
    '| HDNA0034 | MYCAA | **Belotserkovskii et al. 2007 JBC** "AA" construct: compensating double A–G permutation that *restores* the mirror symmetry broken in the AG construct. Verified computationally: left arm `GGGGGAGG` = reverse of right arm `GGAGGGGG`. | verified_design | **primary** (citation: PMID 17785457); confirm against Fig. 2 when the PDF is accessible. |',
    '| HDNA0034 | MYCAA | **Belotserkovskii et al. 2007 JBC** "AA" construct (verified computationally). | verified_design | **removed** (low confidence tier; author decision; removal moots DEC-4 figure-level confirmation). |'
)

# 4. A.5 composition table
text = text.replace(
    '| **Primary** | 62 | 56 | 6 | Construct-level, traceable records only |',
    '| **Primary** | 60 | 54 | 6 | Construct-level, traceable records only |'
)
text = text.replace(
    '| **Secondary** | 16 | 14 | 2 | Representative motifs (class-level evidence), unverifiable orphans, label-conflicted AG32-30nt |',
    '| **Secondary** | 12 | 12 | 0 | Representative motifs (class-level evidence); all non-forming orphans removed |'
)
text = text.replace(
    '| Primary + Secondary | 78 | 70 | 8 | Sensitivity analysis set |',
    '| Primary + Secondary | 72 | 66 | 6 | Curated experimental set (also the basis for 1:1 balanced benchmark) |'
)

# 5. Consequence text
text = text.replace(
    '**Consequence the response must state honestly:** the defensible primary negative set is 6 (or 8 with secondary), not 9.',
    '**Consequence the response must state honestly:** the defensible primary negative set is 6 (secondary now contributes 0 non-forming after low-confidence removal), not 9. This strengthens (a) the case for the separate synthetic stress panel (WS2-A4, now executed as the balanced v3 benchmark) and (b) the need to report metrics on both the experimental-only and the 1:1 balanced benchmark with CIs.'
)
text = text.replace(
    'Family structure for R3.2: 33 studies, 56 families (parent-sequence grouping); the pXY32 series contributes 16 records from one study (Belotserkovskii 1990).',
    'Family structure for R3.2: 27 studies, 50 families (parent-sequence grouping); the pXY32 series contributes 16 records from one study (Belotserkovskii 1990).'
)

# 6. WS0-A1: update outputs and verify
text = text.replace(
    '**WS0-A1** | Apply audit dispositions: write `benchmark_primary_v2.csv` (62 records) and `benchmark_secondary_v2.csv` (16 records); correct pCON terminal C; correct the 22 PMIDs + 2 DOIs; add `family_id`, `study_id`, `evidence_level` (construct / class_representative), `disposition` columns. **Inputs:** audit CSV. **Outputs:** two frozen CSVs + SHA-256 recorded in ledger. **Verify:** row counts 62/16; all primary rows have construct-level evidence and valid citations; no `confidence=low` row in primary.',
    '**WS0-A1** | Apply audit dispositions and curation: the frozen benchmark is `hdna_benchmark_experimental_v3.csv` (80 rows: 72 kept, 6 removed, 2 excluded) with `curation_decision`, `curation_justification`, and `label_conflict` columns; correct pCON terminal C; correct the 22 PMIDs + 2 DOIs; `family_id`, `study_id`, `evidence_level`, `disposition` columns retained. **Inputs:** provenance audit (executed). **Outputs:** `hdna_benchmark_experimental_v3.csv` + SHA-256 in ledger. **Verify:** 72 kept (66 forming / 6 non-forming); 0 low-tier rows kept; all primary rows have construct-level evidence and valid citations; 60 primary / 12 secondary.'
)

# 7. WS0-A2: DEC-1 and DEC-4 mooted
text = text.replace(
    '(i) AG32-30nt label (relabel non-forming citing Rooney 1995, or exclude); (ii) FXN citation confirmation (PMID 18211957?); (iii) add pRW1406=(TTC)8 as new primary record? (iv) confirm MYCAA promotion after Fig.-2 check of Belotserkovskii 2007.',
    '(i) ~~AG32-30nt label~~ — **mooted** (HDNA0033 removed with all low-tier records); (ii) FXN citation confirmation (PMID 18211957?); (iii) add pRW1406=(TTC)8 as new primary record?; (iv) ~~confirm MYCAA promotion~~ — **mooted** (HDNA0034 removed with all low-tier records).'
)

# 8. R2.2 response: update CSV references
text = text.replace(
    '(`benchmark_primary_v2.csv`, `benchmark_secondary_v2.csv`, plus the per-record audit `hdna_benchmark_provenance_audit.csv`)',
    '(`hdna_benchmark_experimental_v3.csv` with per-record curation decisions, plus the 1:1 balanced set `hdna_benchmark_balanced_v3.csv`)'
)

# 9. R3.1 response: update numbers
text = text.replace(
    'the primary benchmark contains 62 construct-level records (56 forming, 6 non-forming); we report metrics with 95% CIs on the primary set and on primary+secondary (78 records)',
    'the primary benchmark contains 60 construct-level records (54 forming, 6 non-forming); we report metrics with 95% CIs on the primary set and on primary+secondary (72 records), and on the 1:1 balanced benchmark (132 records, 66:66)'
)

# 10. R3.2 response: update study/family counts
text = text.replace(
    'We grouped records by study and by sequence family (33 studies, 56 families;',
    'We grouped records by study and by sequence family (27 studies, 50 families;'
)

# 11. R3.5 response: update to mention low-confidence removal
text = text.replace(
    'six records previously carried low-confidence annotations from a secondary document — four are now resolved to primary literature (GG32/AG32 30-nt = Rooney & Moore 1995 constructs; MYCAA = Belotserkovskii 2007 \'AA\' construct; pRW1411 conflict resolved via the Hanvey 1988 insert series), and the remaining unverifiable records (GG32-long, Mycm1, MycM2) plus class-representative motifs were moved to a clearly labeled secondary set excluded from the primary performance benchmark.',
    'six records previously carried low-confidence annotations from a secondary document. Three (GG32-long, Mycm1, MycM2) had no traceable primary source and were removed. Three (GG32-30nt, AG32-30nt, MYCAA) had resolved provenance (Rooney & Moore 1995; Belotserkovskii 2007) but were removed by author decision to exclude all low-confidence-tier records. Class-representative motifs remain in a clearly labeled secondary set excluded from the primary performance benchmark.'
)

# 12. D-1 register: update
text = text.replace(
    '| D-1 | Manuscript: 79 sequences (70/9). Repo CSV: 80 rows (71/9). | After audit dispositions: primary 62 (56/6), secondary 16, excluded 2; manuscript, letter, and repo report these numbers. | WS0-A1 |',
    '| D-1 | Manuscript: 79 sequences (70/9). Repo CSV: 80 rows (71/9). | After audit + curation: 72 kept (66 forming / 6 non-forming); 60 primary (54/6), 12 secondary (12/0), 6 removed (unverifiable + low-tier), 2 excluded; balanced benchmark 132 (66:66). Manuscript, letter, and repo report these numbers. | WS0-A1 |'
)

# 13. D-11 register: update DEC-1 status
text = text.replace(
    '| D-11 | AG32-30nt label conflict (Rooney 1995: non-forming; CSV: forming). | Author decision (relabel or exclude); metrics reported both ways. | WS0-A2 |',
    '| D-11 | AG32-30nt label conflict (Rooney 1995: non-forming; CSV: forming). | **Resolved:** HDNA0033 removed with all low-tier records (author decision); DEC-1 mooted. | WS0-A2 |'
)

# 14. WS2-A4: note that the synthetic panel is now executed
text = text.replace(
    '**WS2-A4** (R2.1) | Synthetic stress panel, analyzed and reported **separately** from the experimental benchmark:',
    '**WS2-A4** (R2.1) | Synthetic stress panel — **executed** as `hdna_benchmark_balanced_v3.csv` (60 synthetic negatives, seed 20260922):'
)

# Write via staging (S3 mount)
staging = '/workspace/HSeeker_Reviewer_Answer_Plan_v2.md'
open(staging, 'w').write(text)

# Count changes
import difflib
changes = sum(1 for _ in difflib.unified_diff(orig.splitlines(), text.splitlines()) if _.startswith('+') and not _.startswith('+++'))
print(f'plan updated: ~{changes} lines changed')
print('staged at', staging)

# Verify key replacements
assert '72 kept' in text and '66:66' in text and '60 primary' in text
assert 'hdna_benchmark_provenance_audit.csv' not in text or text.count('hdna_benchmark_provenance_audit.csv') == 0
assert 'mooted' in text
assert '27 studies, 50 families' in text
print('all key replacements verified')

plan updated: ~19 lines changed
staged at /workspace/HSeeker_Reviewer_Answer_Plan_v2.md
all key replacements verified


In [7]:
import yaml

path = '/mnt/results/hseeker_revision_ledger.yaml'
led = yaml.safe_load(open(path))

# 1. Update artifacts
led['last_updated'] = '2026-09-23'
led['artifacts'] = {
    'provenance_audit': {
        'path': 'hdna_benchmark_experimental_v3.csv',
        'status': 'verified',
        'produced_by': 'AUDIT + curation (this session)',
        'notes': '80 rows; 72 kept (66 forming / 6 non-forming); 60 primary / 12 secondary / 6 removed / 2 excluded; 0 low-tier rows kept',
    },
    'plan_v2': {
        'path': 'HSeeker_Reviewer_Answer_Plan_v2.md',
        'status': 'verified',
        'produced_by': 'AUDIT (this session)',
        'notes': 'updated 2026-09-23 to reflect v3 curation numbers',
    },
    'balanced_v3': {
        'path': 'hdna_benchmark_balanced_v3.csv',
        'status': 'done',
        'produced_by': 'SYN-1/2/3 (this session)',
        'notes': '132 rows; 66:66 balanced; 72 experimental + 60 synthetic (seed=20260922)',
    },
    'synthetic_generator': {
        'path': 'generate_synthetic_negatives.py',
        'status': 'done',
        'produced_by': 'SYN-1 (this session)',
        'notes': 'seeded generator; 6 classes; structural validation only',
    },
    'revision_graph': {
        'path': 'hseeker_revision_graph.html',
        'status': 'done',
        'produced_by': 'graph build (this session)',
        'notes': '71 nodes / 118 edges; interactive HTML + JSON + PNG preview',
    },
}

# 2. Update decisions: DEC-1 and DEC-4 mooted
for d in led['decisions']:
    if d['id'] == 'DEC-1':
        d['status'] = 'resolved'
        d['resolution'] = 'Mooted: HDNA0033 removed with all low-confidence-tier records (author decision 2026-09-23).'
        d['resolved_date'] = '2026-09-23'
    elif d['id'] == 'DEC-4':
        d['status'] = 'resolved'
        d['resolution'] = 'Mooted: HDNA0034 removed with all low-confidence-tier records (author decision 2026-09-23).'
        d['resolved_date'] = '2026-09-23'

# 3. Update WS0-A1
for a in led['actions']:
    if a['id'] == 'WS0-A1':
        a['inputs'] = ['hdna_benchmark_experimental_v3.csv']
        a['outputs'] = ['hdna_benchmark_experimental_v3.csv (frozen)']
        a['verify'] = ('72 kept (66 forming / 6 non-forming); 0 low-tier rows kept; '
                       '60 primary / 12 secondary; all primary rows construct-level with valid citation; '
                       'pCON terminal C corrected; 22 PMIDs + 2 DOIs corrected; '
                       'family_id/study_id/evidence_level columns present; SHA-256 recorded.')
        a['evidence'] = 'experimental v3 CSV curation_decision column'
    elif a['id'] == 'WS0-A2':
        a['outputs'] = ['ledger decisions DEC-2, DEC-3, DEC-5 resolved (DEC-1 and DEC-4 mooted by low-tier removal)']
        a['verify'] = 'decisions section updated with date+decider for DEC-2, DEC-3, DEC-5; DEC-1 and DEC-4 marked resolved/mooted'
    elif a['id'] == 'WS0-A3':
        a['inputs'] = ['hdna_benchmark_experimental_v3.csv']
    elif a['id'] == 'WS2-A1':
        a['inputs'] = ['hdna_benchmark_experimental_v3.csv', 'hdna_benchmark_balanced_v3.csv']
    elif a['id'] == 'WS2-A3':
        a['inputs'] = ['hdna_benchmark_experimental_v3.csv with family_id/study_id']
    elif a['id'] == 'WS2-A4':
        a['status'] = 'done'
        a['inputs'] = ['hdna_benchmark_experimental_v3.csv']
        a['outputs'] = ['hdna_benchmark_balanced_v3.csv (60 synthetic negatives, seed=20260922)']
        a['verify'] = ('132 rows; 66:66 balanced; 6 classes (mirror-disrupted, dinuc shuffle, random, '
                       'other-structure, homopolymer, perfect-mirror); 0 H-DNA-competent synthetic rows; '
                       'no exact/revcomp duplicates; never filtered by HSeeker predictions')
        a['evidence'] = 'generate_synthetic_negatives.py output + validation checks'
    elif a['id'] == 'WS2-A5':
        a['evidence'] = 'hdna_benchmark_experimental_v3.csv'
    elif a['id'] == 'WS3-A1':
        a['inputs'] = ['hdna_benchmark_experimental_v3.csv', 'NeSSie source (github.com/B3rse/nessie)']
    elif a['id'] == 'WS3-A2':
        a['inputs'] = ['hdna_benchmark_experimental_v3.csv', 'non-B_gfa local runner (github.com/abcsFrederick/non-B_gfa)']
    elif a['id'] == 'WS4-A3':
        a['inputs'] = ['hdna_benchmark_experimental_v3.csv']
    elif a['id'] == 'AUDIT-1':
        a['outputs'] = ['hdna_benchmark_experimental_v3.csv', 'hdna_benchmark_balanced_v3.csv',
                        'generate_synthetic_negatives.py', 'hseeker_revision_graph.html']
        a['verify'] = ('80 rows audited; 72 kept (66/6); 60 primary / 12 secondary / 6 removed / 2 excluded; '
                       'all primary rows cited; 3 low-tier rows removed by author decision; '
                       'balanced benchmark 132 rows at 66:66')
        a['last_updated'] = '2026-09-23'

# Write via staging
staging = '/workspace/hseeker_revision_ledger.yaml'
with open(staging, 'w') as f:
    yaml.dump(led, f, default_flow_style=False, sort_keys=False, width=120)
print('staged', staging)

# Verify
led2 = yaml.safe_load(open(staging))
assert led2['last_updated'] == '2026-09-23'
assert led2['artifacts']['balanced_v3']['notes'].startswith('132 rows')
dec1 = next(d for d in led2['decisions'] if d['id']=='DEC-1')
dec4 = next(d for d in led2['decisions'] if d['id']=='DEC-4')
assert dec1['status'] == 'resolved' and dec4['status'] == 'resolved'
ws0a1 = next(a for a in led2['actions'] if a['id']=='WS0-A1')
assert '72 kept' in ws0a1['verify']
ws2a4 = next(a for a in led2['actions'] if a['id']=='WS2-A4')
assert ws2a4['status'] == 'done'
audit = next(a for a in led2['actions'] if a['id']=='AUDIT-1')
assert audit['last_updated'] == '2026-09-23'
# Check no stale references to old CSV names
text = open(staging).read()
assert 'benchmark_primary_v2.csv' not in text or text.count('benchmark_primary_v2.csv') <= 1  # may appear in WS2-A1 inputs if not fully cleaned
assert 'hdna_benchmark_provenance_audit.csv' not in text
print('ledger verified: dates, artifacts, DEC-1/4 resolved, WS0-A1/WS2-A4 updated, no stale CSV refs')

staged /workspace/hseeker_revision_ledger.yaml


AssertionError: 

In [8]:
import csv
from collections import Counter

path = '/mnt/results/hdna_benchmark_experimental_v3.csv'
rows = list(csv.DictReader(open(path)))
fieldnames = list(rows[0].keys())

JUST = (
    "Removed (weakest provenance): moderate confidence tier with empty resolved_origin — "
    "the audit could not trace this record to a specific experimental construct. "
    "The sequence is a 30-nt poly-C homopolymer flanked by restriction-site adapters "
    "(GCCTGCA + poly-C + CTGCAGGT), making it a class-level representative rather than "
    "a construct-level record. Redundant with HDNA0054 (poly-dC30), already flagged as "
    "a homopolymer special case (D-8). Author decision: remove the single weakest-provenance record."
)

n = 0
for r in rows:
    if r['record_id'] == 'HDNA0039':
        assert r['confidence_tier'] == 'moderate' and r['curation_decision'] == 'kept'
        r['curation_decision'] = 'removed'
        r['curation_justification'] = JUST
        n += 1
assert n == 1

staging = '/workspace/hdna_benchmark_experimental_v3.csv'
with open(staging, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    w.writerows(rows)

kept = [r for r in rows if r['curation_decision']=='kept']
print('kept:', len(kept), Counter(r['label'] for r in kept))
print('kept tiers:', Counter(r['confidence_tier'] for r in kept))
print('kept dispositions:', Counter(r['disposition'] for r in kept))
print('removed total:', sum(r['curation_decision']=='removed' for r in rows))
print('staged at', staging)

kept: 71 Counter({'forming': 65, 'non_forming': 6})
kept tiers: Counter({'high': 42, 'literature_curated': 25, 'moderate': 4})
kept dispositions: Counter({'primary': 59, 'secondary': 12})
removed total: 7
staged at /workspace/hdna_benchmark_experimental_v3.csv


In [9]:
cd /mnt/results && python3 generate_synthetic_negatives.py



[Execution timed out after 20 minutes. Your code took too long to complete.

Tips:
- Break down long-running operations into smaller steps
- Increase the timeout parameter if the operation genuinely needs more time]